# Preparation

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/kaggle-llm-science-exam/sample_submission.csv
/kaggle/input/competitions/kaggle-llm-science-exam/train.csv
/kaggle/input/competitions/kaggle-llm-science-exam/test.csv
/kaggle/input/datasets/xhlulu/huggingface-bert/bert-base-multilingual-cased/config.json
/kaggle/input/datasets/xhlulu/huggingface-bert/bert-base-multilingual-cased/tokenizer.json
/kaggle/input/datasets/xhlulu/huggingface-bert/bert-base-multilingual-cased/tf_model.h5
/kaggle/input/datasets/xhlulu/huggingface-bert/bert-base-multilingual-cased/pytorch_model.bin
/kaggle/input/datasets/xhlulu/huggingface-bert/bert-base-multilingual-cased/modelcard.json
/kaggle/input/datasets/xhlulu/huggingface-bert/bert-base-multilingual-cased/vocab.txt
/kaggle/input/datasets/xhlulu/huggingface-bert/bert-large-uncased/config.json
/kaggle/input/datasets/xhlulu/huggingface-bert/bert-large-uncased/tokenizer.json
/kaggle/input/datasets/xhlulu/huggingface-bert/bert-large-uncased/tf_model.h5
/kaggle/input/datasets/xhlulu/h

In [2]:
# Install libraries
import numpy as np
import pandas as pd
import os
import sentencepiece
import sklearn
import string
import torch
import tqdm
import transformers

# Install dependencies
from datasets import Dataset
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModel

# Load the competition dataset

In [3]:
# Examine the data
data = pd.read_csv('/kaggle/input/competitions/kaggle-llm-science-exam/train.csv', index_col='id')

# Generate full column view of the dataset
data

,prompt,A,B,C,D,E,answer
id,,,,,,,
0,Which of the following statements accurately d...,MOND is a theory that reduces the observed mis...,MOND is a theory that increases the discrepanc...,MOND is a theory that explains the missing bar...,MOND is a theory that reduces the discrepancy ...,MOND is a theory that eliminates the observed ...,D
1,Which of the following is an accurate definiti...,Dynamic scaling refers to the evolution of sel...,Dynamic scaling refers to the non-evolution of...,Dynamic scaling refers to the evolution of sel...,Dynamic scaling refers to the non-evolution of...,Dynamic scaling refers to the evolution of sel...,A
2,Which of the following statements accurately d...,The triskeles symbol was reconstructed as a fe...,The triskeles symbol is a representation of th...,The triskeles symbol is a representation of a ...,The triskeles symbol represents three interloc...,The triskeles symbol is a representation of th...,A
3,What is the significance of regularization in ...,Regularizing the mass-energy of an electron wi...,Regularizing the mass-energy of an electron wi...,Regularizing the mass-energy of an electron wi...,Regularizing the mass-energy of an electron wi...,Regularizing the mass-energy of an electron wi...,C
4,Which of the following statements accurately d...,The angular spacing of features in the diffrac...,The angular spacing of features in the diffrac...,The angular spacing of features in the diffrac...,The angular spacing of features in the diffrac...,The angular spacing of features in the diffrac...,D
...,...,...,...,...,...,...,...
195,What is the relation between the three moment ...,The three moment theorem expresses the relatio...,The three moment theorem is used to calculate ...,The three moment theorem describes the relatio...,The three moment theorem is used to calculate ...,The three moment theorem is used to derive the...,C
196,"What is the throttling process, and why is it ...",The throttling process is a steady flow of a f...,The throttling process is a steady adiabatic f...,The throttling process is a steady adiabatic f...,The throttling process is a steady flow of a f...,The throttling process is a steady adiabatic f...,B
197,What happens to excess base metal as a solutio...,"The excess base metal will often solidify, bec...",The excess base metal will often crystallize-o...,"The excess base metal will often dissolve, bec...","The excess base metal will often liquefy, beco...","The excess base metal will often evaporate, be...",B


## Path of the model checkpoint

In [4]:
from transformers import AutoTokenizer

model_dir = '/kaggle/input/datasets/xhlulu/huggingface-bert/bert-base-cased'
tokenizer = AutoTokenizer.from_pretrained(model_dir)

## Convert pandas DataFrame into a dataset

In [5]:
from datasets import Dataset

train_ds = Dataset.from_pandas(data)

# Create a dictionary

In [6]:
# Create a dictionary to convert names into indices
options = 'ABCDE'
indices = list(range(5))

option_to_index = {option: index for option, index in zip(options, indices)}
index_to_option = {index: option for option, index in zip(options, indices)}

def preprocess(example):
    first_sentence = [example['prompt']] * 5
    second_sentence = []
    for option in options:
        second_sentence.append(example[option])

    tokenized_example = tokenizer(first_sentence, second_sentence, truncation=True)
    tokenized_example['label'] = option_to_index[example['answer']]

    return tokenized_example

tokenized_train_ds = train_ds.map(
    preprocess,
    batched=False,
    remove_columns=['prompt', 'A', 'B', 'C', 'D', 'E', 'answer']
)

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

In [7]:
import torch
from dataclasses import dataclass
from typing import Optional, Union
from transformers.tokenization_utils_base import PreTrainedTokenizerBase, PaddingStrategy

@dataclass
class DataCollatorForMultipleChoice:
    tokenizer: PreTrainedTokenizerBase
    padding: Union[bool, str, PaddingStrategy] = True
    max_length: Optional[int] = None
    pad_to_multiple_of: Optional[int] = None

    def __call__(self, features):
        label_name = "label" if 'label' in features[0].keys() else 'labels'
        labels = [feature.pop(label_name) for feature in features]
        batch_size = len(features)
        num_choices = len(features[0]['input_ids'])
        flattened_features = [
            [{k: v[i] for k, v in feature.items()} for i in range(num_choices)] for feature in features
        ]
        flattened_features = sum(flattened_features, [])

        batch = self.tokenizer.pad(
            flattened_features,
            padding=self.padding,
            max_length=self.max_length,
            pad_to_multiple_of=self.pad_to_multiple_of,
            return_tensors='pt'
        )
        batch = {
            k: v.view(batch_size, num_choices, -1) for k, v in batch.items()
        }
        batch['labels'] = torch.tensor(labels, dtype=torch.int64)
        
        return batch

In [8]:
from transformers import AutoModelForMultipleChoice, TrainingArguments, Trainer

model = AutoModelForMultipleChoice.from_pretrained(model_dir)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: /kaggle/input/datasets/xhlulu/huggingface-bert/bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from 

# Model testing

In [9]:
model_dir = 'finetuned_bert'

training_args = TrainingArguments(
    output_dir=model_dir,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    learning_rate=5e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    weight_decay=0.01,
    report_to='none'
)

In [10]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_ds,
    eval_dataset=tokenized_train_ds,
    processing_class=tokenizer,
    data_collator=DataCollatorForMultipleChoice(tokenizer=tokenizer)
)

In [11]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss
1,No log,3.169451
2,No log,2.834361
3,No log,2.729163


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=75, training_loss=3.073952433268229, metrics={'train_runtime': 51.4893, 'train_samples_per_second': 11.653, 'train_steps_per_second': 1.457, 'total_flos': 156159120364080.0, 'train_loss': 3.073952433268229, 'epoch': 3.0})

# Predictions

In [12]:
predictions = trainer.predict(tokenized_train_ds)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


In [13]:
import numpy as np

def predictions_to_map_output(predictions):
    sorted_answer_indices = np.argsort(-predictions)
    top_answer_indices = sorted_answer_indices[:, :3]
    top_answers = np.vectorize(index_to_option.get)(top_answer_indices)

    return np.apply_along_axis(lambda row: ' '.join(row), 1, top_answers)

In [14]:
predictions_to_map_output(predictions.predictions)

array(['D E B', 'A B D', 'A E D', 'B C A', 'C B A', 'B A E', 'A C D',
       'D B A', 'C D E', 'A C E', 'C E A', 'D E C', 'C B A', 'C E D',
       'B E A', 'B C D', 'E B D', 'B E A', 'A D E', 'E C D', 'D B C',
       'B E D', 'C A B', 'C D A', 'E A D', 'E B D', 'A C E', 'A B E',
       'E B A', 'C E B', 'B E C', 'E D C', 'D E B', 'E D B', 'D E B',
       'A D E', 'C D A', 'A B C', 'E D A', 'C E A', 'E D B', 'A C B',
       'B D A', 'D A C', 'D E A', 'A B C', 'B D E', 'C A B', 'B E D',
       'C D E', 'B D C', 'E C A', 'A C B', 'A D C', 'B A E', 'B A E',
       'C D E', 'C B D', 'A C B', 'B C D', 'B E D', 'B E D', 'D A C',
       'C B E', 'B A E', 'E D B', 'C B D', 'D B C', 'D C E', 'D B A',
       'D E A', 'E A D', 'D A B', 'B C A', 'B D C', 'A B E', 'A B C',
       'A E D', 'C A E', 'E A C', 'C D E', 'A B D', 'E C B', 'A E C',
       'C B A', 'D A C', 'A D C', 'A E B', 'C E A', 'B D C', 'D B A',
       'B C D', 'B E C', 'E D B', 'E D B', 'C E B', 'C B D', 'D B E',
       'C D B', 'E C

In [15]:
data_test = pd.read_csv('/kaggle/input/competitions/kaggle-llm-science-exam/test.csv')

data_test

,id,prompt,A,B,C,D,E
0,0,Which of the following statements accurately d...,MOND is a theory that reduces the observed mis...,MOND is a theory that increases the discrepanc...,MOND is a theory that explains the missing bar...,MOND is a theory that reduces the discrepancy ...,MOND is a theory that eliminates the observed ...
1,1,Which of the following is an accurate definiti...,Dynamic scaling refers to the evolution of sel...,Dynamic scaling refers to the non-evolution of...,Dynamic scaling refers to the evolution of sel...,Dynamic scaling refers to the non-evolution of...,Dynamic scaling refers to the evolution of sel...
2,2,Which of the following statements accurately d...,The triskeles symbol was reconstructed as a fe...,The triskeles symbol is a representation of th...,The triskeles symbol is a representation of a ...,The triskeles symbol represents three interloc...,The triskeles symbol is a representation of th...
3,3,What is the significance of regularization in ...,Regularizing the mass-energy of an electron wi...,Regularizing the mass-energy of an electron wi...,Regularizing the mass-energy of an electron wi...,Regularizing the mass-energy of an electron wi...,Regularizing the mass-energy of an electron wi...
4,4,Which of the following statements accurately d...,The angular spacing of features in the diffrac...,The angular spacing of features in the diffrac...,The angular spacing of features in the diffrac...,The angular spacing of features in the diffrac...,The angular spacing of features in the diffrac...
...,...,...,...,...,...,...,...
195,195,What is the relation between the three moment ...,The three moment theorem expresses the relatio...,The three moment theorem is used to calculate ...,The three moment theorem describes the relatio...,The three moment theorem is used to calculate ...,The three moment theorem is used to derive the...
196,196,"What is the throttling process, and why is it ...",The throttling process is a steady flow of a f...,The throttling process is a steady adiabatic f...,The throttling process is a steady adiabatic f...,The throttling process is a steady flow of a f...,The throttling process is a steady adiabatic f...
197,197,What happens to excess base metal as a solutio...,"The excess base metal will often solidify, bec...",The excess base metal will often crystallize-o...,"The excess base metal will often dissolve, bec...","The excess base metal will often liquefy, beco...","The excess base metal will often evaporate, be..."
198,198,"What is the relationship between mass, force, ...",Mass is a property that determines the weight ...,Mass is an inertial property that determines a...,Mass is an inertial property that determines a...,Mass is an inertial property that determines a...,Mass is a property that determines the size of...


In [16]:
data_test['answer'] = 'A'

test_ds = Dataset.from_pandas(data_test)
tokenized_test_ds = test_ds.map(
    preprocess,
    batched=False,
    remove_columns=['prompt', 'A', 'B', 'C', 'D', 'E', 'answer']
)

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

In [17]:
test_predictions = trainer.predict(tokenized_test_ds)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


# File submission

In [18]:
submission_df = data_test[['id']]
submission_df['prediction'] = predictions_to_map_output(test_predictions.predictions)

submission_df.head()

/tmp/ipykernel_22/832998074.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  submission_df['prediction'] = predictions_to_map_output(test_predictions.predictions)


,id,prediction
0,0,D E B
1,1,A B D
2,2,A E D
3,3,B C A
4,4,C B A


In [19]:
submission_df.to_csv('submission.csv', index=False)
print("Successfully saved as CSV file.")

Successfully saved as CSV file.


In [20]:
def map_at_3(predictions, true_answers):
    top_3_predictions = predictions_to_map_output(predictions.predictions)

    average_precisions = []
    for i in range(len(true_answers)):
        true_answer = true_answers[i]
        true_answer = options[true_answer]
        predicted_answers = top_3_predictions[i].split(" ")

        if true_answer in predicted_answers:
            index_of_true_answer = predicted_answers.index(true_answer)
            precision_at_index = 1 / (index_of_true_answer + 1)
            average_precisions.append(precision_at_index)
        else:
            average_precisions.append(0)

        map_3 = np.mean(average_precisions)
        return map_3

true_answers = tokenized_train_ds['label']
map_3_score = map_at_3(predictions, true_answers)
print(f"MAP@3 Score: {map_3_score}")

MAP@3 Score: 1.0


In [21]:
os.environ["CUDA_VISIBLE_DEVICES"] = "0, 1"

sim_model = 'all-MiniLM-L6-v2'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
batch_size = 500_000

In [22]:
qna_df = pd.read_csv('/kaggle/input/competitions/kaggle-llm-science-exam/train.csv', index_col=0)

qna_df.head()

,prompt,A,B,C,D,E,answer
id,,,,,,,
0,Which of the following statements accurately d...,MOND is a theory that reduces the observed mis...,MOND is a theory that increases the discrepanc...,MOND is a theory that explains the missing bar...,MOND is a theory that reduces the discrepancy ...,MOND is a theory that eliminates the observed ...,D
1,Which of the following is an accurate definiti...,Dynamic scaling refers to the evolution of sel...,Dynamic scaling refers to the non-evolution of...,Dynamic scaling refers to the evolution of sel...,Dynamic scaling refers to the non-evolution of...,Dynamic scaling refers to the evolution of sel...,A
2,Which of the following statements accurately d...,The triskeles symbol was reconstructed as a fe...,The triskeles symbol is a representation of th...,The triskeles symbol is a representation of a ...,The triskeles symbol represents three interloc...,The triskeles symbol is a representation of th...,A
3,What is the significance of regularization in ...,Regularizing the mass-energy of an electron wi...,Regularizing the mass-energy of an electron wi...,Regularizing the mass-energy of an electron wi...,Regularizing the mass-energy of an electron wi...,Regularizing the mass-energy of an electron wi...,C
4,Which of the following statements accurately d...,The angular spacing of features in the diffrac...,The angular spacing of features in the diffrac...,The angular spacing of features in the diffrac...,The angular spacing of features in the diffrac...,The angular spacing of features in the diffrac...,D
